In [ ]:
# ============================================================
# 全タイムステップ：
# x-z平面 Density + normalized (vx, vz) direction
#
# 表示単位
#   距離：AU
#   時間：Myr
#   密度：M_sun AU^-3
#
# ・y=0と交差するAMRブロックのみ使用
# ・各ブロックからy=0に最も近い1層を抽出
# ・AMR境界線なし
# ・速度は補間後に規格化し、方向のみ表示
# ・密度カラースケール計算ではシンク内部を除外
# ============================================================

import os
import re
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pyvista as pv
from matplotlib.colors import LogNorm
from scipy.interpolate import griddata


# ============================================================
# 入出力設定
# ============================================================
vtk_dir = Path(
    os.path.expanduser(
        "~/athena-project/results/Toyouchi-test27"
    )
).resolve()

output_dir = Path(
    "./xz_density_normalized_velocity"
).resolve()

output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

# 使用するVTK出力ストリーム
STREAM_TOKEN = ".out2."

print(f"[INFO] Input : {vtk_dir}")
print(f"[INFO] Output: {output_dir}")

if not vtk_dir.is_dir():
    raise FileNotFoundError(
        f"VTK directory does not exist: {vtk_dir}"
    )


# ============================================================
# 単位
# ============================================================
M_UNIT_CGS = 4.0e33
L_UNIT_CGS = 6.7e15
T_UNIT_CGS = 3.34e10

AU_CGS = 1.495978707e13
MSUN_CGS = 1.98847e33
MYR_CGS = (
    1.0e6
    * 365.25
    * 24.0
    * 3600.0
)

LENGTH_UNIT_AU = L_UNIT_CGS / AU_CGS
TIME_UNIT_MYR = T_UNIT_CGS / MYR_CGS
MASS_UNIT_MSUN = M_UNIT_CGS / MSUN_CGS

DENSITY_UNIT_MSUN_AU3 = (
    MASS_UNIT_MSUN
    / LENGTH_UNIT_AU**3
)

print(
    f"[INFO] 1 code length  = "
    f"{LENGTH_UNIT_AU:.6e} AU"
)
print(
    f"[INFO] 1 code time    = "
    f"{TIME_UNIT_MYR:.6e} Myr"
)
print(
    f"[INFO] 1 code density = "
    f"{DENSITY_UNIT_MSUN_AU3:.6e} "
    "M_sun AU^-3"
)


# ============================================================
# 描画設定
# ============================================================

PLOT_RADIUS_CODE = 500.0

PLOT_RADIUS_AU = (
    PLOT_RADIUS_CODE
    * LENGTH_UNIT_AU
)

X_MIN_AU = -PLOT_RADIUS_AU
X_MAX_AU = PLOT_RADIUS_AU
Z_MIN_AU = -PLOT_RADIUS_AU
Z_MAX_AU = PLOT_RADIUS_AU

# 補間画像の解像度
MAP_RESOLUTION = 500

# 矢印数
QUIVER_N = 31

# 全時刻共通の密度範囲
DENSITY_PERCENTILES = (1.0, 99.5)

# 密度範囲を求めるときに除外するシンク領域
SINK_RADIUS_AU = 1000.0

# 極端に遅い速度では矢印を表示しない
MIN_SPEED_FRACTION = 0.01

DENSITY_CMAP = "Greys"
QUIVER_COLOR = "black"

FIGSIZE = (8, 8)
DPI = 200


# ============================================================
# 変数名候補
# ============================================================
DENSITY_CANDIDATES = (
    "rho",
    "dens",
    "density",
    "prim_dens",
    "prim_density",
)

VELOCITY_CANDIDATES = (
    "vel",
    "velocity",
    "v",
)

MOMENTUM_CANDIDATES = (
    "mom",
    "momentum",
)


# ============================================================
# VTKヘッダからcode timeを取得
# ============================================================
def read_vtk_time_code(filename):
    with open(filename, "rb") as vtk_file:
        header = vtk_file.read(2048).decode(
            "ascii",
            errors="ignore",
        )

    match = re.search(
        r"time\s*=\s*"
        r"([+-]?(?:\d+\.?\d*|\.\d+)"
        r"(?:[eE][+-]?\d+)?)",
        header,
        flags=re.IGNORECASE,
    )

    if match is None:
        raise ValueError(
            f"VTK header time not found: {filename}"
        )

    return float(match.group(1))


# ============================================================
# out2番号を取得
# ============================================================
def get_step_number(filename):
    name = Path(filename).name

    match = re.search(
        r"(?:prim\.)?out2\.(\d+)",
        name,
    )

    if match is None:
        return None

    return int(match.group(1))


# ============================================================
# point_dataの変数名検索
# ============================================================
def find_point_field(dataset, candidates):
    lower_to_original = {
        name.lower(): name
        for name in dataset.point_data.keys()
    }

    for candidate in candidates:
        key = candidate.lower()

        if key in lower_to_original:
            return lower_to_original[key]

    return None


# ============================================================
# 正確なy=0面を抽出
# ============================================================
def extract_block_xz_slice(filename):
    """
    セル中心の最近傍面ではなく、PyVistaのsliceを使って
    幾何学的に正確なy=0面を抽出する。
    """

    grid = pv.read(filename)

    xmin, xmax, ymin, ymax, zmin, zmax = grid.bounds

    bounds_tolerance = (
        1.0e-12
        * max(
            abs(ymin),
            abs(ymax),
            1.0,
        )
    )

    # y=0と交差しないブロックを除外
    if not (
        ymin - bounds_tolerance
        <= 0.0
        <= ymax + bounds_tolerance
    ):
        return None

    # cell dataをpoint dataへ補間
    point_grid = grid.cell_data_to_point_data(
        pass_cell_data=False,
    )

    # 正確なy=0面
    sliced = point_grid.slice(
        normal=(0.0, 1.0, 0.0),
        origin=(0.0, 0.0, 0.0),
    )

    if sliced.n_points == 0:
        return None

    density_name = find_point_field(
        sliced,
        DENSITY_CANDIDATES,
    )

    if density_name is None:
        raise KeyError(
            f"Density field not found after slice: "
            f"{filename}\n"
            f"point_data="
            f"{list(sliced.point_data.keys())}"
        )

    density_code = np.asarray(
        sliced.point_data[density_name]
    ).reshape(-1)

    velocity_name = find_point_field(
        sliced,
        VELOCITY_CANDIDATES,
    )

    if velocity_name is not None:
        velocity = np.asarray(
            sliced.point_data[velocity_name]
        )

    else:
        momentum_name = find_point_field(
            sliced,
            MOMENTUM_CANDIDATES,
        )

        if momentum_name is None:
            raise KeyError(
                f"Velocity/momentum field not found: "
                f"{filename}\n"
                f"point_data="
                f"{list(sliced.point_data.keys())}"
            )

        momentum = np.asarray(
            sliced.point_data[momentum_name]
        )

        velocity = momentum / np.maximum(
            density_code[:, None],
            1.0e-300,
        )

    if (
        velocity.ndim != 2
        or velocity.shape[1] < 3
    ):
        raise ValueError(
            f"Unexpected velocity shape: "
            f"{velocity.shape}"
        )

    points = np.asarray(sliced.points)

    x_au = points[:, 0] * LENGTH_UNIT_AU
    z_au = points[:, 2] * LENGTH_UNIT_AU

    density = (
        density_code
        * DENSITY_UNIT_MSUN_AU3
    )

    vx = velocity[:, 0]
    vz = velocity[:, 2]

    valid = (
        np.isfinite(x_au)
        & np.isfinite(z_au)
        & np.isfinite(density)
        & (density > 0.0)
        & np.isfinite(vx)
        & np.isfinite(vz)
        & (x_au >= X_MIN_AU)
        & (x_au <= X_MAX_AU)
        & (z_au >= Z_MIN_AU)
        & (z_au <= Z_MAX_AU)
    )

    if not np.any(valid):
        return None

    return {
        "x": x_au[valid],
        "z": z_au[valid],
        "rho": density[valid],
        "vx": vx[valid],
        "vz": vz[valid],
    }


# ============================================================
# AMRブロック境界の重複座標を統合
# ============================================================
def merge_duplicate_points(data):
    """
    同じ(x,z)に複数のAMRブロックから値が入った場合に統合する。

    密度は対数平均、速度は算術平均を使用する。
    """

    x = np.asarray(data["x"])
    z = np.asarray(data["z"])
    rho = np.asarray(data["rho"])
    vx = np.asarray(data["vx"])
    vz = np.asarray(data["vz"])

    coordinate_tolerance = max(
        PLOT_RADIUS_AU * 1.0e-10,
        1.0e-8,
    )

    ix = np.rint(
        x / coordinate_tolerance
    ).astype(np.int64)

    iz = np.rint(
        z / coordinate_tolerance
    ).astype(np.int64)

    keys = np.column_stack((ix, iz))

    _, inverse = np.unique(
        keys,
        axis=0,
        return_inverse=True,
    )

    count = np.bincount(
        inverse
    ).astype(float)

    x_merged = (
        np.bincount(inverse, weights=x)
        / count
    )

    z_merged = (
        np.bincount(inverse, weights=z)
        / count
    )

    log_rho_merged = (
        np.bincount(
            inverse,
            weights=np.log10(
                np.maximum(rho, 1.0e-300)
            ),
        )
        / count
    )

    rho_merged = 10.0**log_rho_merged

    vx_merged = (
        np.bincount(inverse, weights=vx)
        / count
    )

    vz_merged = (
        np.bincount(inverse, weights=vz)
        / count
    )

    return {
        "x": x_merged,
        "z": z_merged,
        "rho": rho_merged,
        "vx": vx_merged,
        "vz": vz_merged,
    }


# ============================================================
# 1スナップショット分のx-z面を作る
# ============================================================
def extract_snapshot(step_info):
    collected = {
        "x": [],
        "z": [],
        "rho": [],
        "vx": [],
        "vz": [],
    }

    n_used_blocks = 0

    for filename in step_info["files"]:
        try:
            block_data = extract_block_xz_slice(
                filename
            )

            if block_data is None:
                continue

            n_used_blocks += 1

            for key in collected:
                collected[key].append(
                    block_data[key]
                )

        except Exception as error:
            print(
                f"[WARNING] {Path(filename).name}: "
                f"{error}"
            )

    if not collected["x"]:
        raise RuntimeError(
            f"No x-z slice data at "
            f"step={step_info['step']:05d}"
        )

    raw_data = {
        key: np.concatenate(values)
        for key, values in collected.items()
    }

    merged_data = merge_duplicate_points(
        raw_data
    )

    merged_data["step"] = step_info["step"]
    merged_data["time_code"] = step_info[
        "time_code"
    ]
    merged_data["time_myr"] = step_info[
        "time_myr"
    ]

    print(
        f"[INFO] step={step_info['step']:05d}: "
        f"blocks={n_used_blocks}/"
        f"{len(step_info['files'])}, "
        f"raw points={len(raw_data['x']):,}, "
        f"merged points="
        f"{len(merged_data['x']):,}"
    )

    return merged_data


# ============================================================
# 線形補間＋領域外のみnearest補完
# ============================================================
def interpolate_field(
    points_2d,
    values,
    X,
    Z,
):
    values = np.asarray(values)

    linear = griddata(
        points_2d,
        values,
        (X, Z),
        method="linear",
    )

    if np.any(~np.isfinite(linear)):
        nearest = griddata(
            points_2d,
            values,
            (X, Z),
            method="nearest",
        )

        result = np.where(
            np.isfinite(linear),
            linear,
            nearest,
        )
    else:
        result = linear

    return result


# ============================================================
# VTKファイルを探索
# 同一ディレクトリだけを対象にし、別のrunとの混在を防ぐ
# ============================================================
vtk_files = sorted(
    path
    for path in vtk_dir.glob("*.vtk")
    if "Toyouchi.block" in path.name
)

if STREAM_TOKEN is not None:
    vtk_files = [
        path
        for path in vtk_files
        if STREAM_TOKEN in path.name
    ]

if not vtk_files:
    raise FileNotFoundError(
        f"No VTK files found in {vtk_dir}"
    )

print(
    f"[INFO] VTK file count: "
    f"{len(vtk_files)}"
)


# ============================================================
# out2番号ごとにブロックをまとめる
# ============================================================
files_by_step = defaultdict(list)

for filename in vtk_files:
    step = get_step_number(filename)

    if step is not None:
        files_by_step[step].append(filename)

if not files_by_step:
    raise RuntimeError(
        "No out2 timestep numbers were found."
    )


# ============================================================
# スナップショット一覧
# 同じstep内の時刻は中央値を使用
# ============================================================
step_table = []

for step in sorted(files_by_step):
    files = sorted(files_by_step[step])

    block_times = np.array(
        [
            read_vtk_time_code(filename)
            for filename in files
        ],
        dtype=float,
    )

    time_code = float(
        np.nanmedian(block_times)
    )

    time_spread = float(
        np.nanmax(block_times)
        - np.nanmin(block_times)
    )

    allowed_spread = max(
        1.0e-10,
        abs(time_code) * 1.0e-10,
    )

    if time_spread > allowed_spread:
        print(
            f"[WARNING] step={step:05d}: "
            f"block times differ by "
            f"{time_spread:.6e} code time"
        )

    step_table.append({
        "step": step,
        "time_code": time_code,
        "time_myr": (
            time_code
            * TIME_UNIT_MYR
        ),
        "files": files,
    })

# 物理時間順
step_table.sort(
    key=lambda item: (
        item["time_code"],
        item["step"],
    )
)

print(
    f"[INFO] Snapshot count: "
    f"{len(step_table)}"
)
print(
    f"[INFO] First time: "
    f"{step_table[0]['time_myr']:.6e} Myr"
)
print(
    f"[INFO] Last time : "
    f"{step_table[-1]['time_myr']:.6e} Myr"
)


# ============================================================
# 全スナップショットのx-z面を抽出
# ============================================================
all_snapshots = []

for index, step_info in enumerate(step_table):
    print(
        f"[INFO] Reading "
        f"{index + 1}/{len(step_table)}: "
        f"step={step_info['step']:05d}, "
        f"t={step_info['time_myr']:.6e} Myr"
    )

    try:
        snapshot = extract_snapshot(
            step_info
        )

        all_snapshots.append(snapshot)

    except Exception as error:
        print(
            f"[WARNING] step="
            f"{step_info['step']:05d}: "
            f"{error}"
        )

if not all_snapshots:
    raise RuntimeError(
        "No valid x-z snapshots were extracted."
    )


# ============================================================
# 全時刻共通の密度カラースケール
# シンク内部は統計から除外
# ============================================================
density_samples = []

for snapshot in all_snapshots:
    radius_au = np.hypot(
        snapshot["x"],
        snapshot["z"],
    )

    mask = (
        np.isfinite(snapshot["rho"])
        & (snapshot["rho"] > 0.0)
        & (radius_au >= SINK_RADIUS_AU)
    )

    values = snapshot["rho"][mask]

    if values.size > 0:
        # 大量データ対策として各時刻最大20万点
        if values.size > 200000:
            sample_index = np.linspace(
                0,
                values.size - 1,
                200000,
                dtype=int,
            )
            values = values[sample_index]

        density_samples.append(values)

if not density_samples:
    raise RuntimeError(
        "No positive density values were found "
        "outside the sink."
    )

global_density = np.concatenate(
    density_samples
)

rho_vmin, rho_vmax = np.percentile(
    global_density,
    DENSITY_PERCENTILES,
)

if (
    not np.isfinite(rho_vmin)
    or not np.isfinite(rho_vmax)
    or rho_vmin <= 0.0
    or rho_vmax <= rho_vmin
):
    raise RuntimeError(
        "Invalid global density color range: "
        f"{rho_vmin}, {rho_vmax}"
    )

density_norm = LogNorm(
    vmin=rho_vmin,
    vmax=rho_vmax,
    clip=True,
)

print(
    f"[INFO] Global density range: "
    f"{rho_vmin:.6e} -- "
    f"{rho_vmax:.6e} M_sun AU^-3"
)


# ============================================================
# 固定補間グリッド
# ============================================================
x_axis = np.linspace(
    X_MIN_AU,
    X_MAX_AU,
    MAP_RESOLUTION,
)

z_axis = np.linspace(
    Z_MIN_AU,
    Z_MAX_AU,
    MAP_RESOLUTION,
)

X_map, Z_map = np.meshgrid(
    x_axis,
    z_axis,
)


# ============================================================
# 全時刻を描画
# ============================================================
saved_files = []

for frame_index, snapshot in enumerate(
    all_snapshots
):
    print(
        f"[INFO] Plotting "
        f"{frame_index + 1}/"
        f"{len(all_snapshots)}: "
        f"step={snapshot['step']:05d}"
    )

    points_xz = np.column_stack(
        (
            snapshot["x"],
            snapshot["z"],
        )
    )

    # 密度は対数値を補間
    # 正の量を通常値で線形補間するより安定
    log_rho_map = interpolate_field(
        points_xz,
        np.log10(
            np.maximum(
                snapshot["rho"],
                1.0e-300,
            )
        ),
        X_map,
        Z_map,
    )

    rho_map = 10.0**log_rho_map

    vx_map = interpolate_field(
        points_xz,
        snapshot["vx"],
        X_map,
        Z_map,
    )

    vz_map = interpolate_field(
        points_xz,
        snapshot["vz"],
        X_map,
        Z_map,
    )

    # --------------------------------------------------------
    # 補間後に速度を規格化
    # --------------------------------------------------------
    speed_map = np.hypot(
        vx_map,
        vz_map,
    )

    finite_speed = speed_map[
        np.isfinite(speed_map)
        & (speed_map > 0.0)
    ]

    if finite_speed.size > 0:
        representative_speed = np.median(
            finite_speed
        )

        minimum_speed = (
            representative_speed
            * MIN_SPEED_FRACTION
        )
    else:
        minimum_speed = np.inf

    valid_velocity = (
        np.isfinite(vx_map)
        & np.isfinite(vz_map)
        & np.isfinite(speed_map)
        & (speed_map > minimum_speed)
    )

    unit_vx = np.full_like(
        vx_map,
        np.nan,
    )

    unit_vz = np.full_like(
        vz_map,
        np.nan,
    )

    unit_vx[valid_velocity] = (
        vx_map[valid_velocity]
        / speed_map[valid_velocity]
    )

    unit_vz[valid_velocity] = (
        vz_map[valid_velocity]
        / speed_map[valid_velocity]
    )

    # --------------------------------------------------------
    # 矢印を間引く
    # --------------------------------------------------------
    quiver_indices = np.linspace(
        0,
        MAP_RESOLUTION - 1,
        QUIVER_N,
        dtype=int,
    )

    selection = np.ix_(
        quiver_indices,
        quiver_indices,
    )

    X_quiver = X_map[selection]
    Z_quiver = Z_map[selection]
    U_quiver = unit_vx[selection]
    W_quiver = unit_vz[selection]

    # 矢印長
    arrow_length_au = (
        1.25
        * min(
            X_MAX_AU - X_MIN_AU,
            Z_MAX_AU - Z_MIN_AU,
        )
        / max(QUIVER_N - 1, 1)
    )

    # --------------------------------------------------------
    # 描画
    # --------------------------------------------------------
    fig, ax = plt.subplots(
        figsize=FIGSIZE,
    )

    ax.pcolormesh(
        X_map,
        Z_map,
        np.clip(
            rho_map,
            rho_vmin,
            rho_vmax,
        ),
        shading="auto",
        cmap=DENSITY_CMAP,
        norm=density_norm,
        rasterized=True,
    )

    ax.quiver(
        X_quiver,
        Z_quiver,
        U_quiver * arrow_length_au,
        W_quiver * arrow_length_au,
        color=QUIVER_COLOR,
        angles="xy",
        scale_units="xy",
        scale=1.0,
        width=0.0025,
        pivot="mid",
        headwidth=3.5,
        headlength=4.5,
        headaxislength=4.0,
        alpha=0.9,
    )

    ax.axhline(
        0.0,
        color="gray",
        linewidth=0.7,
        alpha=0.7,
    )

    ax.axvline(
        0.0,
        color="gray",
        linewidth=0.7,
        alpha=0.7,
    )

    ax.plot(
        0.0,
        0.0,
        marker="+",
        color="red",
        markersize=12,
        markeredgewidth=2.5,
    )

    ax.set_xlim(
        X_MIN_AU,
        X_MAX_AU,
    )

    ax.set_ylim(
        Z_MIN_AU,
        Z_MAX_AU,
    )

    ax.set_xlabel(
        r"$x\ [{\rm AU}]$"
    )

    ax.set_ylabel(
        r"$z\ [{\rm AU}]$"
    )

    ax.set_title(
        "Density + normalized "
        r"$(v_x,v_z)$ direction"
        "\n"
        f"t = "
        f"{snapshot['time_myr']:.6e} Myr",
        fontsize=14,
    )

    ax.set_aspect("equal")
    ax.grid(False)

    # カラーバーと凡例なし
    fig.tight_layout()

    output_file = output_dir / (
        f"frame_{frame_index:05d}_"
        f"step_{snapshot['step']:05d}_"
        f"time_{snapshot['time_myr']:.6e}Myr.png"
    )

    fig.savefig(
        output_file,
        dpi=DPI,
        bbox_inches="tight",
        facecolor="white",
    )

    plt.close(fig)

    saved_files.append(output_file)

    print(
        f"[INFO] Saved: "
        f"{output_file.name}"
    )


print(
    f"\n[INFO] Completed: "
    f"{len(saved_files)} figures"
)
print(
    f"[INFO] Output directory: "
    f"{output_dir}"
)

In [ ]:
# ============================================================
# 全タイムステップ：
# x-y平面 Density + normalized (vx, vy) direction
#
# 表示単位
#   距離：AU
#   時間：Myr
#   密度：M_sun AU^-3
#
# ちらつき・AMRアーティファクト対策
#   1. PyVista sliceで正確なz=0面を抽出
#   2. 同じ(x,y)に重複したAMRデータを統合
#   3. 全時刻で共通の密度カラースケールを使用
#   4. 密度は対数値で補間
#   5. out2番号単位でAMRブロックをまとめる
#   6. 物理時間順に画像を保存
# ============================================================

import os
import re
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pyvista as pv
from matplotlib.colors import LogNorm
from matplotlib.ticker import FuncFormatter
from scipy.interpolate import griddata


# ============================================================
# 入出力設定
# ============================================================
vtk_dir = Path(
    os.path.expanduser(
        "~/athena-project/results/〇〇"
    )
).resolve()

output_dir = (
    vtk_dir
    / "xy_density_normalized_velocity"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

STREAM_TOKEN = ".out2."

print(f"[INFO] Input : {vtk_dir}")
print(f"[INFO] Output: {output_dir}")

if not vtk_dir.is_dir():
    raise FileNotFoundError(
        f"VTK directory does not exist: {vtk_dir}"
    )


# ============================================================
# 単位
# ============================================================
M_UNIT_CGS = 4.0e33
L_UNIT_CGS = 6.7e15
T_UNIT_CGS = 3.34e10

AU_CGS = 1.495978707e13
MSUN_CGS = 1.98847e33

MYR_CGS = (
    1.0e6
    * 365.25
    * 24.0
    * 3600.0
)

LENGTH_UNIT_AU = (
    L_UNIT_CGS
    / AU_CGS
)

TIME_UNIT_MYR = (
    T_UNIT_CGS
    / MYR_CGS
)

MASS_UNIT_MSUN = (
    M_UNIT_CGS
    / MSUN_CGS
)

DENSITY_UNIT_MSUN_AU3 = (
    MASS_UNIT_MSUN
    / LENGTH_UNIT_AU**3
)

print(
    f"[INFO] 1 code length  = "
    f"{LENGTH_UNIT_AU:.6e} AU"
)
print(
    f"[INFO] 1 code time    = "
    f"{TIME_UNIT_MYR:.6e} Myr"
)
print(
    f"[INFO] 1 code density = "
    f"{DENSITY_UNIT_MSUN_AU3:.6e} "
    "M_sun AU^-3"
)


# ============================================================
# 描画設定
# ============================================================
PLOT_RADIUS_AU = 1.0e5

MAP_RESOLUTION = 700
QUIVER_N = 31

# 全時刻の密度分布から使用する範囲
DENSITY_PERCENTILES = (
    5.0,
    99.0,
)

# カラースケールの統計からシンク内部を除外
SINK_RADIUS_AU = 1000.0

# 極端に遅い領域では矢印を非表示
MIN_SPEED_FRACTION = 0.01

DENSITY_CMAP = "Greys"
QUIVER_COLOR = "black"

FIGSIZE = (8, 8)
DPI = 200


# ============================================================
# フィールド名候補
# ============================================================
DENSITY_CANDIDATES = (
    "rho",
    "dens",
    "density",
    "prim_dens",
    "prim_density",
)

VELOCITY_CANDIDATES = (
    "vel",
    "velocity",
    "v",
)

MOMENTUM_CANDIDATES = (
    "mom",
    "momentum",
)


# ============================================================
# VTK時刻
# ============================================================
def read_vtk_time_code(filename):
    with open(filename, "rb") as vtk_file:
        header = vtk_file.read(2048).decode(
            "ascii",
            errors="ignore",
        )

    match = re.search(
        r"time\s*=\s*"
        r"([+-]?(?:\d+\.?\d*|\.\d+)"
        r"(?:[eE][+-]?\d+)?)",
        header,
        flags=re.IGNORECASE,
    )

    if match is None:
        raise ValueError(
            f"VTK time not found: {filename}"
        )

    return float(match.group(1))


def get_step_number(filename):
    match = re.search(
        r"(?:prim\.)?out2\.(\d+)",
        Path(filename).name,
    )

    if match is None:
        return None

    return int(match.group(1))


# ============================================================
# フィールド検索
# ============================================================
def find_cell_field(grid, candidates):
    lower_to_original = {
        name.lower(): name
        for name in grid.cell_data.keys()
    }

    for candidate in candidates:
        key = candidate.lower()

        if key in lower_to_original:
            return lower_to_original[key]

    return None


def extract_density_velocity(grid, filename):
    density_name = find_cell_field(
        grid,
        DENSITY_CANDIDATES,
    )

    if density_name is None:
        raise KeyError(
            f"Density field not found: {filename}\n"
            f"cell_data="
            f"{list(grid.cell_data.keys())}"
        )

    density = np.asarray(
        grid.cell_data[density_name]
    ).reshape(-1)

    velocity_name = find_cell_field(
        grid,
        VELOCITY_CANDIDATES,
    )

    if velocity_name is not None:
        velocity = np.asarray(
            grid.cell_data[velocity_name]
        )

    else:
        momentum_name = find_cell_field(
            grid,
            MOMENTUM_CANDIDATES,
        )

        if momentum_name is None:
            raise KeyError(
                f"Velocity/momentum field not found: "
                f"{filename}"
            )

        momentum = np.asarray(
            grid.cell_data[momentum_name]
        )

        velocity = (
            momentum
            / np.maximum(
                density[:, None],
                1.0e-300,
            )
        )

    if (
        velocity.ndim != 2
        or velocity.shape[1] < 3
    ):
        raise ValueError(
            f"Unexpected velocity shape: "
            f"{velocity.shape}"
        )

    return density, velocity[:, :3]


# ============================================================
# 各ブロックから正確なz=0面を抽出
# ============================================================
def extract_xy_midplane(filename):
    grid = pv.read(filename)

    xmin, xmax, ymin, ymax, zmin, zmax = (
        grid.bounds
    )

    tolerance = (
        1.0e-12
        * max(
            abs(zmin),
            abs(zmax),
            1.0,
        )
    )

    if not (
        zmin - tolerance
        <= 0.0
        <= zmax + tolerance
    ):
        return None

    density_code, velocity = (
        extract_density_velocity(
            grid,
            filename,
        )
    )

    working_grid = grid.copy()

    working_grid.cell_data[
        "__analysis_density__"
    ] = density_code

    working_grid.cell_data[
        "__analysis_velocity__"
    ] = velocity

    # セル中心値を点データへ補間
    point_grid = (
        working_grid.cell_data_to_point_data(
            pass_cell_data=False,
        )
    )

    # 幾何学的に正確なz=0面
    sliced = point_grid.slice(
        normal=(0.0, 0.0, 1.0),
        origin=(0.0, 0.0, 0.0),
    )

    if sliced.n_points == 0:
        return None

    points_code = np.asarray(
        sliced.points
    )

    density_slice = np.asarray(
        sliced.point_data[
            "__analysis_density__"
        ]
    ).reshape(-1)

    velocity_slice = np.asarray(
        sliced.point_data[
            "__analysis_velocity__"
        ]
    )

    x_au = (
        points_code[:, 0]
        * LENGTH_UNIT_AU
    )

    y_au = (
        points_code[:, 1]
        * LENGTH_UNIT_AU
    )

    density = (
        density_slice
        * DENSITY_UNIT_MSUN_AU3
    )

    vx = velocity_slice[:, 0]
    vy = velocity_slice[:, 1]

    valid = (
        np.isfinite(x_au)
        & np.isfinite(y_au)
        & np.isfinite(density)
        & (density > 0.0)
        & np.isfinite(vx)
        & np.isfinite(vy)
        & (np.abs(x_au) <= PLOT_RADIUS_AU)
        & (np.abs(y_au) <= PLOT_RADIUS_AU)
    )

    if not np.any(valid):
        return None

    return {
        "x": x_au[valid],
        "y": y_au[valid],
        "rho": density[valid],
        "vx": vx[valid],
        "vy": vy[valid],
    }


# ============================================================
# AMRブロック境界の重複点を統合
# ============================================================
def merge_duplicate_points(data):
    x = np.asarray(data["x"])
    y = np.asarray(data["y"])
    rho = np.asarray(data["rho"])
    vx = np.asarray(data["vx"])
    vy = np.asarray(data["vy"])

    coordinate_tolerance = max(
        PLOT_RADIUS_AU * 1.0e-10,
        1.0e-8,
    )

    ix = np.rint(
        x / coordinate_tolerance
    ).astype(np.int64)

    iy = np.rint(
        y / coordinate_tolerance
    ).astype(np.int64)

    coordinate_keys = np.column_stack(
        (ix, iy)
    )

    _, inverse = np.unique(
        coordinate_keys,
        axis=0,
        return_inverse=True,
    )

    counts = np.bincount(
        inverse
    ).astype(float)

    x_merged = (
        np.bincount(
            inverse,
            weights=x,
        )
        / counts
    )

    y_merged = (
        np.bincount(
            inverse,
            weights=y,
        )
        / counts
    )

    # 密度は対数平均
    log_rho_merged = (
        np.bincount(
            inverse,
            weights=np.log10(
                np.maximum(
                    rho,
                    1.0e-300,
                )
            ),
        )
        / counts
    )

    rho_merged = (
        10.0**log_rho_merged
    )

    # 速度は算術平均
    vx_merged = (
        np.bincount(
            inverse,
            weights=vx,
        )
        / counts
    )

    vy_merged = (
        np.bincount(
            inverse,
            weights=vy,
        )
        / counts
    )

    return {
        "x": x_merged,
        "y": y_merged,
        "rho": rho_merged,
        "vx": vx_merged,
        "vy": vy_merged,
    }


# ============================================================
# 1タイムステップ分を抽出
# ============================================================
def extract_snapshot(step_info):
    collected = {
        "x": [],
        "y": [],
        "rho": [],
        "vx": [],
        "vy": [],
    }

    used_blocks = 0

    for filename in step_info["files"]:
        try:
            block_data = (
                extract_xy_midplane(filename)
            )

            if block_data is None:
                continue

            used_blocks += 1

            for key in collected:
                collected[key].append(
                    block_data[key]
                )

        except Exception as error:
            print(
                f"[WARNING] "
                f"{Path(filename).name}: "
                f"{error}"
            )

    if not collected["x"]:
        raise RuntimeError(
            f"No x-y data at "
            f"step={step_info['step']:05d}"
        )

    raw_data = {
        key: np.concatenate(values)
        for key, values in collected.items()
    }

    merged = merge_duplicate_points(
        raw_data
    )

    merged["step"] = step_info["step"]
    merged["time_code"] = step_info[
        "time_code"
    ]
    merged["time_myr"] = step_info[
        "time_myr"
    ]

    print(
        f"[INFO] step={step_info['step']:05d}: "
        f"blocks={used_blocks}/"
        f"{len(step_info['files'])}, "
        f"raw={len(raw_data['x']):,}, "
        f"merged={len(merged['x']):,}"
    )

    return merged


# ============================================================
# 補間
# ============================================================
def interpolate_field(
    points_2d,
    values,
    X,
    Y,
):
    linear = griddata(
        points_2d,
        values,
        (X, Y),
        method="linear",
    )

    if np.any(~np.isfinite(linear)):
        nearest = griddata(
            points_2d,
            values,
            (X, Y),
            method="nearest",
        )

        return np.where(
            np.isfinite(linear),
            linear,
            nearest,
        )

    return linear


# ============================================================
# VTKファイル探索
# 同一フォルダだけを対象にする
# ============================================================
vtk_files = sorted(
    path
    for path in vtk_dir.glob("*.vtk")
    if "Toyouchi.block" in path.name
)

if STREAM_TOKEN is not None:
    vtk_files = [
        path
        for path in vtk_files
        if STREAM_TOKEN in path.name
    ]

if not vtk_files:
    raise FileNotFoundError(
        f"No VTK files found in {vtk_dir}"
    )

print(
    f"[INFO] VTK file count: "
    f"{len(vtk_files)}"
)


# ============================================================
# out2番号ごとにグループ化
# ============================================================
files_by_step = defaultdict(list)

for filename in vtk_files:
    step = get_step_number(filename)

    if step is not None:
        files_by_step[step].append(filename)

if not files_by_step:
    raise RuntimeError(
        "No out2 timestep numbers were found."
    )


# ============================================================
# スナップショット一覧
# ============================================================
step_table = []

for step in sorted(files_by_step):
    files = sorted(
        files_by_step[step]
    )

    block_times = np.array(
        [
            read_vtk_time_code(filename)
            for filename in files
        ],
        dtype=float,
    )

    time_code = float(
        np.nanmedian(block_times)
    )

    time_spread = float(
        np.nanmax(block_times)
        - np.nanmin(block_times)
    )

    allowed_spread = max(
        1.0e-10,
        abs(time_code) * 1.0e-10,
    )

    if time_spread > allowed_spread:
        print(
            f"[WARNING] step={step:05d}: "
            f"block-time spread="
            f"{time_spread:.6e} code time"
        )

    step_table.append({
        "step": step,
        "time_code": time_code,
        "time_myr": (
            time_code
            * TIME_UNIT_MYR
        ),
        "files": files,
    })

# 物理時間順
step_table.sort(
    key=lambda item: (
        item["time_code"],
        item["step"],
    )
)

print(
    f"[INFO] Snapshot count: "
    f"{len(step_table)}"
)
print(
    f"[INFO] Time range: "
    f"{step_table[0]['time_myr']:.6e} -- "
    f"{step_table[-1]['time_myr']:.6e} Myr"
)


# ============================================================
# 第1段階：
# 全時刻のx-y面を抽出し、密度範囲を決定
#
# この段階ではまだPNGは作られない。
# ============================================================
snapshots = []
density_samples = []

print(
    "\n[INFO] Pass 1/2: "
    "scanning global density range"
)

for index, step_info in enumerate(
    step_table
):
    print(
        f"[INFO] Scan "
        f"{index + 1}/{len(step_table)}: "
        f"step={step_info['step']:05d}"
    )

    try:
        snapshot = extract_snapshot(
            step_info
        )

    except Exception as error:
        print(
            f"[WARNING] step="
            f"{step_info['step']:05d}: "
            f"{error}"
        )
        continue

    snapshots.append(snapshot)

    radius_au = np.hypot(
        snapshot["x"],
        snapshot["y"],
    )

    density_mask = (
        np.isfinite(snapshot["rho"])
        & (snapshot["rho"] > 0.0)
        & (radius_au >= SINK_RADIUS_AU)
    )

    values = snapshot["rho"][
        density_mask
    ]

    if values.size > 0:
        # 時刻ごとの点数差による重み付けを抑える
        maximum_sample_count = 100000

        if values.size > maximum_sample_count:
            sample_indices = np.linspace(
                0,
                values.size - 1,
                maximum_sample_count,
                dtype=int,
            )

            values = values[
                sample_indices
            ]

        density_samples.append(values)

if not snapshots:
    raise RuntimeError(
        "No valid x-y snapshots were extracted."
    )

if not density_samples:
    raise RuntimeError(
        "No positive density values were found "
        "outside the sink."
    )

global_density = np.concatenate(
    density_samples
)

rho_vmin, rho_vmax = np.percentile(
    global_density,
    DENSITY_PERCENTILES,
)

if (
    not np.isfinite(rho_vmin)
    or not np.isfinite(rho_vmax)
    or rho_vmin <= 0.0
    or rho_vmax <= rho_vmin
):
    raise RuntimeError(
        "Invalid global density range: "
        f"{rho_vmin}, {rho_vmax}"
    )

density_norm = LogNorm(
    vmin=rho_vmin,
    vmax=rho_vmax,
    clip=True,
)

print(
    "\n[INFO] Global density range:"
)
print(
    f"       {rho_vmin:.6e} -- "
    f"{rho_vmax:.6e} M_sun AU^-3"
)


# ============================================================
# 固定補間グリッド
# ============================================================
map_axis = np.linspace(
    -PLOT_RADIUS_AU,
    PLOT_RADIUS_AU,
    MAP_RESOLUTION,
)

X_map, Y_map = np.meshgrid(
    map_axis,
    map_axis,
)

quiver_indices = np.linspace(
    0,
    MAP_RESOLUTION - 1,
    QUIVER_N,
    dtype=int,
)

quiver_selection = np.ix_(
    quiver_indices,
    quiver_indices,
)

arrow_length_au = (
    1.25
    * 2.0
    * PLOT_RADIUS_AU
    / max(QUIVER_N - 1, 1)
)


# ============================================================
# 第2段階：
# 共通カラースケールで順次描画・保存
# ============================================================
saved_files = []

print(
    "\n[INFO] Pass 2/2: "
    "plotting and saving PNG files"
)

for frame_index, snapshot in enumerate(
    snapshots
):
    print(
        f"[INFO] Plot "
        f"{frame_index + 1}/{len(snapshots)}: "
        f"step={snapshot['step']:05d}"
    )

    points_xy = np.column_stack(
        (
            snapshot["x"],
            snapshot["y"],
        )
    )

    # 密度はlog10で補間
    log_rho_map = interpolate_field(
        points_xy,
        np.log10(
            np.maximum(
                snapshot["rho"],
                1.0e-300,
            )
        ),
        X_map,
        Y_map,
    )

    rho_map = (
        10.0**log_rho_map
    )

    vx_map = interpolate_field(
        points_xy,
        snapshot["vx"],
        X_map,
        Y_map,
    )

    vy_map = interpolate_field(
        points_xy,
        snapshot["vy"],
        X_map,
        Y_map,
    )

    # --------------------------------------------------------
    # 補間後に速度を規格化
    # --------------------------------------------------------
    speed_map = np.hypot(
        vx_map,
        vy_map,
    )

    finite_speed = speed_map[
        np.isfinite(speed_map)
        & (speed_map > 0.0)
    ]

    if finite_speed.size > 0:
        minimum_speed = (
            np.median(finite_speed)
            * MIN_SPEED_FRACTION
        )
    else:
        minimum_speed = np.inf

    valid_velocity = (
        np.isfinite(vx_map)
        & np.isfinite(vy_map)
        & np.isfinite(speed_map)
        & (speed_map > minimum_speed)
    )

    unit_vx = np.full_like(
        vx_map,
        np.nan,
    )

    unit_vy = np.full_like(
        vy_map,
        np.nan,
    )

    unit_vx[valid_velocity] = (
        vx_map[valid_velocity]
        / speed_map[valid_velocity]
    )

    unit_vy[valid_velocity] = (
        vy_map[valid_velocity]
        / speed_map[valid_velocity]
    )

    X_quiver = X_map[
        quiver_selection
    ]

    Y_quiver = Y_map[
        quiver_selection
    ]

    U_quiver = unit_vx[
        quiver_selection
    ]

    V_quiver = unit_vy[
        quiver_selection
    ]

    # --------------------------------------------------------
    # 描画
    # --------------------------------------------------------
    fig, ax = plt.subplots(
        figsize=FIGSIZE,
    )

    ax.pcolormesh(
        X_map,
        Y_map,
        np.clip(
            rho_map,
            rho_vmin,
            rho_vmax,
        ),
        shading="auto",
        cmap=DENSITY_CMAP,
        norm=density_norm,
        rasterized=True,
    )

    ax.quiver(
        X_quiver,
        Y_quiver,
        U_quiver * arrow_length_au,
        V_quiver * arrow_length_au,
        color=QUIVER_COLOR,
        angles="xy",
        scale_units="xy",
        scale=1.0,
        width=0.0025,
        pivot="mid",
        headwidth=3.5,
        headlength=4.5,
        headaxislength=4.0,
        alpha=0.9,
    )

    ax.axhline(
        0.0,
        color="gray",
        linewidth=0.7,
        alpha=0.7,
    )

    ax.axvline(
        0.0,
        color="gray",
        linewidth=0.7,
        alpha=0.7,
    )

    ax.plot(
        0.0,
        0.0,
        marker="+",
        color="red",
        markersize=12,
        markeredgewidth=2.5,
    )

    ax.set_xlim(
        -PLOT_RADIUS_AU,
        PLOT_RADIUS_AU,
    )

    ax.set_ylim(
        -PLOT_RADIUS_AU,
        PLOT_RADIUS_AU,
    )

    # 実際の座標はAUのまま、目盛り表示だけ10^4 AU単位に変換
    axis_scale_au = 1.0e4

    axis_formatter = FuncFormatter(
        lambda value, position: f"{value / axis_scale_au:g}"
    )

    ax.xaxis.set_major_formatter(
        axis_formatter
    )

    ax.yaxis.set_major_formatter(
        axis_formatter
    )

    ax.set_xlabel(
        r"$x\ [10^4\ {\rm AU}]$"
    )

    ax.set_ylabel(
        r"$y\ [10^4\ {\rm AU}]$"
    )

    ax.set_title(
        "Density + normalized "
        r"$(v_x,v_y)$ direction"
        "\n"
        f"t = "
        f"{snapshot['time_myr']:.6e} Myr",
        fontsize=14,
    )

    ax.set_aspect("equal")
    ax.grid(False)

    fig.tight_layout()

    output_file = output_dir / (
        f"frame_{frame_index:05d}_"
        f"step_{snapshot['step']:05d}_"
        f"time_{snapshot['time_myr']:.6e}Myr.png"
    )

    fig.savefig(
        output_file,
        dpi=DPI,
        bbox_inches="tight",
        facecolor="white",
    )

    plt.close(fig)

    saved_files.append(
        output_file
    )

    print(
        f"[INFO] Saved: "
        f"{output_file.name}"
    )


print(
    f"\n[INFO] Completed: "
    f"{len(saved_files)} PNG files"
)
print(
    f"[INFO] Output directory: "
    f"{output_dir}"
)
print(
    f"[INFO] Existing PNG count: "
    f"{len(list(output_dir.glob('*.png')))}"
)